# SkyLens — 영상 AI 학습 노트북

4채널(RGB+열화상) 인지 모델을 학습한다. 단일 백본 + 이중 헤드:

| 헤드 | 출력 | 대상 |
|---|---|---|
| 세그멘테이션 | per-pixel 클래스 맵 | 위험구역 (stuff: 화재·붕괴·도로차단) |
| 점 검출 (CenterNet) | 중심점 히트맵 + (w,h) | 사람 (인스턴스) |

> **설계 근거**: [`src/skylens_model/README.md`](src/skylens_model/README.md) — 왜 UNet인지,
> 왜 modality dropout인지, 왜 사람이 점인지 전부 여기 기록돼 있다.
> **데이터셋 조사**: [`docs/DATASETS.md`](docs/DATASETS.md)

학습 도중 언제 꺼도 되도록 구성돼 있다 (§6 중단·재개).


## 1. 환경 확인

`skylens_model` 은 `uv sync` 로 설치된다. 노트북은 `uv run jupyter lab` 으로 띄운다:
```bash
uv sync
```

In [ ]:
import sys, warnings
import numpy as np
import torch

warnings.filterwarnings("ignore")

import skylens_model

print(f"python       {sys.version.split()[0]}")
print(f"torch        {torch.__version__}")
print(f"skylens      {skylens_model.__version__}  ({skylens_model.__file__})")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device       {DEVICE}", f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")

## 2. 설정

`IMAGE_SIZE`는 32의 배수여야 한다(백본이 5단계 다운샘플). 사람이 초소형 객체라
(SARD 기준 면적 0.1% 미만) 해상도를 과하게 줄이면 안 된다 — README §1.5 참조.

In [ ]:
from pathlib import Path

DATA_ROOT = Path("data")          # 데이터셋 루트 (git 제외됨)
OUTPUT_DIR = Path("runs/skylens") # 체크포인트/로그 (git 제외됨)

IMAGE_SIZE = 512                  # 32의 배수. 모든 데이터셋을 이 크기로 리사이즈한다
PERSON_HEAD_STRIDE = 4            # 히트맵 해상도 = IMAGE_SIZE / 4
NUM_DANGER_CLASSES = 4            # 0=정상 1=화재 2=붕괴 3=도로차단

BACKBONE = "microsoft/resnet-50"
BATCH_SIZE = 4                    # RTX 4050(6GB)에서 512px 기준 VRAM 1.7GB
EPOCHS = 1
LR = 1e-4
SAVE_EVERY_STEPS = 300            # 이 스텝마다 체크포인트 (중단 대비, 약 1분 간격)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"image {IMAGE_SIZE}x{IMAGE_SIZE} -> heatmap {IMAGE_SIZE // PERSON_HEAD_STRIDE}^2")


## 3. 데이터셋

실제로 내려받아 둔 데이터만 사용한다. 각 데이터셋은 **자기가 라벨을 가진 헤드만** 학습시킨다
(README §6.3 헤드별 분리 학습) — 재난과 사람이 동시에 라벨링된 공개 데이터는 없기 때문이다.

| 데이터셋 | 경로 | 모달리티 | 학습시키는 헤드 |
|---|---|---|---|
| **LLVIP** | `data/llvip/LLVIP` | RGB + 열화상 | 점 검출 (사람) · **4채널 Early Fusion 검증** |
| **RescueNet** | `data/rescuenet` | RGB | 세그 (붕괴·도로차단) |
| **RescueNet-roads** | `data/rescuenet_roads` | RGB | 세그 (도로차단만) |
| **VisDrone** | `data/visdrone` | RGB | 점 검출 (드론 시점 워밍업) |

> 없는 데이터셋은 자동으로 건너뛴다. 다운로드 절차는
> [`src/skylens_model/datasets/README.md`](src/skylens_model/datasets/README.md) 참조.


In [ ]:
import albumentations as A
import numpy as np


def build_transform(size: int, train: bool = True):
    """이미지·세그 마스크·bbox 를 한 번에 변환한다.

    데이터셋마다 원본 해상도가 달라(LLVIP 1024x1280, RescueNet 3000x4000 등)
    collator 가 균일 크기를 요구하므로 리사이즈는 필수다.
    albumentations 를 쓰는 이유는 4채널 이미지 + 마스크 + bbox 의 좌표계를
    한 번에 맞춰 주기 때문이다 (README §7.3).
    """
    ops = [A.Resize(size, size)]
    if train:
        ops += [A.HorizontalFlip(p=0.5), A.RandomBrightnessContrast(p=0.2)]
    tf = A.Compose(
        ops,
        bbox_params=A.BboxParams(format="pascal_voc", label_fields=["cls"], min_visibility=0.2),
    )

    def apply(sample: dict) -> dict:
        boxes = sample.get("person_boxes")
        has_boxes = boxes is not None and len(boxes) > 0
        mask = sample.get("danger_mask")
        kw = {
            "image": sample["image"],
            "bboxes": boxes.tolist() if has_boxes else [],
            "cls": [0] * (len(boxes) if has_boxes else 0),
        }
        # 마스크가 없는 데이터셋은 mask 인자를 아예 넘기지 않는다
        if mask is not None:
            kw["mask"] = mask
        out = tf(**kw)

        sample = dict(sample)
        sample["image"] = out["image"]
        sample["danger_mask"] = out["mask"] if mask is not None else None
        if sample.get("person_boxes") is not None:
            b = out["bboxes"]
            sample["person_boxes"] = (
                np.asarray(b, dtype=np.float32) if b else np.zeros((0, 4), np.float32)
            )
        return sample

    return apply


train_tf = build_transform(IMAGE_SIZE, train=True)
eval_tf = build_transform(IMAGE_SIZE, train=False)
print("transform 준비 완료")


### 3.1 데이터셋 조립

내려받힌 것만 골라 `ConcatDataset` 으로 묶는다. 서로 다른 헤드를 학습시키는
데이터가 한 배치에 섞여도 되며(모델이 GT 있는 헤드의 loss만 계산한다),
모달리티가 달라도 된다(대칭 modality dropout 이 흡수한다 — README §2.2).


In [ ]:
from torch.utils.data import ConcatDataset
from skylens_model.datasets import (
    LLVIP,
    RescueNetSegmentation,
    SARD,
    VisDronePerson,
)

# (클래스, 경로, 생성자 인자) — 없는 것은 건너뛴다
SOURCES = [
    (LLVIP, DATA_ROOT / "llvip" / "LLVIP", {}),
    (RescueNetSegmentation, DATA_ROOT / "rescuenet", {}),
    (VisDronePerson, DATA_ROOT / "visdrone", {}),
    (SARD, DATA_ROOT / "sard", {}),
]


def build_split(split: str, transform):
    parts = []
    for cls, root, kw in SOURCES:
        if not root.exists():
            print(f"  [없음] {cls.__name__:24s} {root}")
            continue
        try:
            ds = cls(root, split=split, transforms=transform, **kw)
            print(f"  [ok]   {cls.__name__:24s} {len(ds):>6,}장")
            parts.append(ds)
        except Exception as e:
            print(f"  [skip] {cls.__name__:24s} {type(e).__name__}: {str(e).splitlines()[0][:70]}")
    return ConcatDataset(parts) if parts else None


print("[train]")
train_ds = build_split("train", train_tf)
print("[eval]")
eval_ds = build_split("val", eval_tf) or build_split("test", eval_tf)

if train_ds is None:
    raise RuntimeError(
        "학습 데이터가 없다. src/skylens_model/datasets/README.md 의 다운로드 절차를 따를 것."
    )
print(f"
train {len(train_ds):,} | eval {len(eval_ds):,}" if eval_ds else f"
train {len(train_ds):,} | eval 없음")


## 4. Collator — 타겟 인코딩

raw 샘플(`image`/`danger_mask`/`person_boxes`)을 모델 입력으로 변환한다.

- **bbox → 중심점 가우시안 히트맵** (CenterNet). bbox 라벨을 버리지 않고
  중심은 히트맵으로, 크기는 `wh` 회귀 타깃으로 쓴다 — README §1.4.
- **모달리티 결손 인코딩**: 열화상을 `[0.1, 1.0]`으로 정규화하고 `0.0`을
  "없음" 전용으로 예약한다. radiometric thermal은 0이 유효 온도일 수 있어
  단순 zero-fill로는 "없음"과 "차갑다"를 구분 못 한다 — README §2.3.

> ⚠️ **modality dropout은 모델이 담당한다** (`SkyLensConfig`의 확률).
> collator에도 같은 기능이 있지만 이중 적용을 피하려고 기본값(꺼짐)으로 둔다.

In [ ]:
from skylens_model.utils import SkyLensCollator

collator = SkyLensCollator(
    person_head_stride=PERSON_HEAD_STRIDE,
    validity_channel=False,
    modality_dropout=(0.0, 0.0),  # 모델이 담당 (이중 적용 방지)
)

# 계약 확인 — 세그 샘플 + 사람 샘플을 섞은 배치
_batch = collator([train_ds[0], train_ds[1]])
for k, v in sorted(_batch.items()):
    print(f"  {k:16s} {str(tuple(v.shape)):24s} {v.dtype}")
print("\nmodality_mask [rgb, thermal]:", _batch["modality_mask"].tolist())
print("사람 중심점 개수:", int(_batch["person_reg_mask"].sum()))

## 5. 모델

`transformers` 표준 규약을 따른다 — `SkyLensConfig` + `SkyLensForDisasterPerception`.
인코더는 `AutoBackbone`(CNN)이고, UNet 디코더와 이중 헤드는 자체 구현이다.

**4채널 inflation**: 사전학습 첫 conv를 확장하되 RGB 3채널은 그대로 보존하고,
4번째(열화상) 채널은 RGB 가중치의 평균으로 채운다. 0이나 랜덤보다 초기
활성 스케일이 유지돼 안정적이다 — README §6.1.

In [ ]:
from skylens_model.models import SkyLensConfig, SkyLensForDisasterPerception

config = SkyLensConfig(
    backbone=BACKBONE,
    use_pretrained_backbone=True,
    in_channels=4,
    num_danger_classes=NUM_DANGER_CLASSES,
    person_head_stride=PERSON_HEAD_STRIDE,
    # 대칭 modality dropout — RGB만/열화상만/둘다 를 한 모델이 모두 커버 (README §2.2)
    modality_dropout_rgb_only=0.25,
    modality_dropout_thermal_only=0.25,
    seg_loss_weight=1.0,
    heatmap_loss_weight=1.0,
    wh_loss_weight=0.1,
)

model = SkyLensForDisasterPerception(config)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"파라미터 {n_params:.1f}M")

# 초기화 건전성 — heatmap 예측이 0.1 근처여야 focal loss가 정상 범위에서 시작한다
model.train()
with torch.no_grad():
    _o = model(pixel_values=torch.randn(2, 4, IMAGE_SIZE, IMAGE_SIZE))
print(f"초기 heatmap 평균 {float(_o.person_heatmap.mean()):.4f}  (목표 0.10, bias=-2.19)")
assert 0.05 < float(_o.person_heatmap.mean()) < 0.2, "헤드 초기화 이상 — focal loss가 폭주한다"
print("초기화 OK")

## 6. 학습

`SkyLensTrainer`는 HF `Trainer`를 상속한다. 추가된 것:

- 개별 loss 컴포넌트 로깅 (`loss_danger_seg` / `loss_person_heatmap` / `loss_person_wh`)
- `freeze_backbone_epochs` — 초반 백본 동결 워밍업
- `remove_unused_columns=False` 강제 — collator가 raw 샘플을 변환하는 구조라 필수

### 중단·재개

학습 도중 꺼야 하는 일이 잦으므로 **언제 꺼도 잃는 게 없게** 구성했다.

| 장치 | 동작 |
|---|---|
| `save_steps=SAVE_EVERY_STEPS` | N스텝마다 체크포인트 (에폭 단위면 중단 시 통째로 날아감) |
| `GracefulInterruptCallback` | **Ctrl+C·커널 인터럽트·SIGTERM** 수신 시 현재 스텝을 마치고 저장 후 정상 종료 |
| `find_resume_checkpoint` | 학습 셀을 다시 실행하면 마지막 체크포인트에서 자동 재개 |

체크포인트에는 가중치뿐 아니라 **optimizer·scheduler·RNG state·global_step**이
함께 저장되므로, 재개하면 학습률 스케줄과 데이터 순서까지 끊긴 지점에서 이어진다.
즉시 죽이지 않고 스텝 경계에서 저장하는 이유는 저장 도중 종료되어 체크포인트가
깨지는 것을 막기 위해서다. 급하면 **Ctrl+C를 한 번 더** 누르면 즉시 중단된다.

> **프리트레인 전략** (README §6.1): ImageNet → **VisDrone 워밍업** → 본 학습.
> ImageNet(지상 일반 사진)에서 재난 데이터로 바로 점프하면 도메인 갭이 크므로,
> VisDrone이 "드론 시점 + 초소형 사람"이라는 중간 다리를 놓는다.


In [ ]:
from skylens_model.utils.trainer import SkyLensTrainer
from skylens_model.utils.training_args import SkyLensTrainingArguments
from skylens_model.utils.metrics import build_compute_metrics
from skylens_model.utils.callbacks import GracefulInterruptCallback, find_resume_checkpoint

args = SkyLensTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    # --- 중단·재개 전제 설정 ---------------------------------------------
    # 에폭 단위로만 저장하면 중간에 끄는 순간 그 에폭이 통째로 날아간다.
    # 스텝 단위로 자주 저장해 두고, save_total_limit 으로 디스크를 관리한다.
    save_strategy="steps",
    save_steps=SAVE_EVERY_STEPS,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=SAVE_EVERY_STEPS,
    load_best_model_at_end=False,  # 중단이 잦으므로 '마지막'을 기준으로 이어간다
    save_safetensors=True,
    # ---------------------------------------------------------------------
    report_to=[],
    fp16=(DEVICE == "cuda"),
    dataloader_num_workers=0,
    # SkyLens 전용
    person_head_stride=PERSON_HEAD_STRIDE,
    num_danger_classes=NUM_DANGER_CLASSES,
    freeze_backbone_epochs=0.0,   # 워밍업이 필요하면 0.5~1.0
    eval_score_threshold=0.3,
    point_distance_threshold=8.0,
)

# Ctrl+C / 커널 인터럽트 / SIGTERM 을 받으면
# 현재 스텝을 마치고 체크포인트를 저장한 뒤 정상 종료한다.
interrupt_cb = GracefulInterruptCallback()

trainer = SkyLensTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    callbacks=[interrupt_cb],
    compute_metrics=build_compute_metrics(
        num_classes=NUM_DANGER_CLASSES,
        distance_threshold=args.point_distance_threshold,
        score_threshold=args.eval_score_threshold,
    ),
)
print("Trainer 준비 완료 —", SAVE_EVERY_STEPS, "스텝마다 체크포인트 저장")


In [ ]:
# 이전에 중단된 체크포인트가 있으면 자동으로 이어서 학습한다.
# 처음부터 다시 하려면 OUTPUT_DIR 를 비우거나 resume=None 으로 둘 것.
resume = find_resume_checkpoint(OUTPUT_DIR)
print(f"재개 지점: {resume}" if resume else "체크포인트 없음 — 처음부터 학습")

result = trainer.train(resume_from_checkpoint=resume)

print()
if interrupt_cb.interrupted:
    print(f"[중단됨] global_step={result.global_step} 까지 저장 완료.")
    print("이 셀을 다시 실행하면 그 지점에서 이어서 학습한다.")
else:
    print(f"[완료] train_loss {result.training_loss:.4f} | steps {result.global_step}")


### 손실 곡선

세 컴포넌트(`danger_seg` / `person_heatmap` / `person_wh`)가 **각각** 내려가는지 본다.
합산 loss만 보면 한 헤드가 무너지는 걸 놓친다.


In [ ]:
import matplotlib.pyplot as plt

hist = [h for h in trainer.state.log_history if "loss" in h]
if hist:
    keys = [k for k in ("loss", "loss_danger_seg", "loss_person_heatmap", "loss_person_wh") if k in hist[0]]
    fig, ax = plt.subplots(figsize=(8, 4))
    for k in keys:
        xs = [h["step"] for h in hist if k in h]
        ys = [h[k] for h in hist if k in h]
        ax.plot(xs, ys, marker="o", ms=3, label=k)
    ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
    ax.set_title("SkyLens training loss")
    plt.tight_layout(); plt.show()
else:
    print("로그가 없다 — logging_steps 를 줄여볼 것")

## 7. 평가

- **세그**: mIoU / per-class IoU / pixel accuracy (ignore=255 제외)
- **점 검출**: 예측 점과 GT 점의 **거리 기반 매칭**으로 precision/recall/F1.
  bbox IoU가 아니라 점 거리를 쓰는 이유는 최종 산출물이 점이기 때문이다 — README §1.4.

In [ ]:
metrics = trainer.evaluate()
for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f"  {k:32s} {v:.4f}")

## 8. 추론 — 히트맵 → 점

학습된 히트맵에서 실제 탐지 좌표를 뽑는 단계. 3×3 max-pool NMS로 로컬 피크를
찾고 top-k를 취한다.

여기서 나오는 `(x, y)`가 **파이프라인의 다음 단계인 Depth Map 레이캐스팅의 입력**이다
— 이 점 하나가 3D 세계좌표로 역투영되어 3D 상황판의 마커가 된다.
`(w, h)`는 그 지점의 depth를 robust하게 샘플링하기 위한 영역으로 쓴다(README §1.4).

In [ ]:
from skylens_model.utils.metrics import decode_heatmap_peaks

model.eval()
sample_batch = collator([eval_ds[1]])  # 홀수 = 사람 샘플
with torch.no_grad():
    out = model(pixel_values=sample_batch["pixel_values"].to(model.device))

dets = decode_heatmap_peaks(
    out.person_heatmap.cpu(), out.person_wh.cpu(),
    k=20, threshold=0.2, stride=PERSON_HEAD_STRIDE,
)[0]
kept = dets[dets[:, 4] > 0]
print(f"탐지 {len(kept)}건  (학습이 안 된 모델이면 0건이 정상)")
for x, y, w, h, s in kept[:5].tolist():
    print(f"  point=({x:7.1f},{y:7.1f})  size=({w:5.1f},{h:5.1f})  score={s:.3f}")

seg_pred = out.danger_logits.argmax(1)[0].cpu().numpy()
uniq, cnt = np.unique(seg_pred, return_counts=True)
print("\n세그 예측 클래스 분포:", dict(zip(uniq.tolist(), cnt.tolist())))

## 9. 저장

`save_pretrained`로 저장하면 `from_pretrained`로 그대로 복원된다
(transformers 표준 규약을 지킨 덕분).

In [ ]:
SAVE_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(SAVE_DIR))
print(f"저장 → {SAVE_DIR}")

# 왕복 검증
reloaded = SkyLensForDisasterPerception.from_pretrained(str(SAVE_DIR))
reloaded.eval(); model.eval()
with torch.no_grad():
    x = torch.randn(1, 4, IMAGE_SIZE, IMAGE_SIZE)
    a = model(pixel_values=x.to(model.device)).person_heatmap.cpu()
    b = reloaded(pixel_values=x).person_heatmap
print("복원 일치:", torch.allclose(a, b, atol=1e-5))

## 10. 다음 단계

1. **데이터 보강** — 현재 확보: LLVIP(사람·4채널), RescueNet(붕괴·도로차단).
   미확보: FLAME(화재 세그), SARD(SAR 자세 사람), FLAME 3(4채널 재난).
   화재 클래스(1)는 아직 학습 데이터가 없다.
2. **VisDrone 워밍업** — 본 학습 전 백본을 드론 시점에 적응시킨다 (README §6.1)
3. **추론 결과를 뷰어에 연결** — 모델 출력을
   `src/skylens_core/protocol.ts` 의 `DetectionResult` 로 직렬화되어 RECON 화면의
   마커가 된다. 이 단계가 PROJECT.md §7의 "연출된 부분"을 "실제 추론"으로 바꾸는 지점이다.

> 미결정 사항은 [`src/skylens_model/README.md`](src/skylens_model/README.md) §9에 정리돼 있다.
